In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime
from tqdm import tqdm
import os

In [2]:
def get_schedule(year):
    url = f"https://www.espn.com/nba/team/schedule/_/name/cha/season/{year}"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"Failed to fetch schedule for {year}")
            return []
        
        soup = BeautifulSoup(response.content, 'html.parser')
        # Find all game links
        # Links look like /nba/game/_/gameId/401584698
        links = soup.find_all('a', href=re.compile(r'/nba/game/_/gameId/\d+'))
        game_ids = set()
        for link in links:
            match = re.search(r'gameId/(\d+)', link['href'])
            if match:
                game_ids.add(match.group(1))
        
        return list(game_ids)
    except Exception as e:
        print(f"Error getting schedule for {year}: {e}")
        return []

# Test
ids_2024 = get_schedule(2024)
print(f"Found {len(ids_2024)} games for 2024")

Found 82 games for 2024


In [3]:
ARENA_CAPACITIES = {
    # Current NBA Arenas (as of 2023-24)
    "State Farm Arena": 16600,       # Atlanta Hawks
    "TD Garden": 19156,              # Boston Celtics
    "Barclays Center": 17732,        # Brooklyn Nets
    "Spectrum Center": 19077,        # Charlotte Hornets
    "United Center": 20917,          # Chicago Bulls
    "Rocket Mortgage FieldHouse": 19432, # Cleveland Cavaliers
    "American Airlines Center": 19200, # Dallas Mavericks
    "Ball Arena": 19605,             # Denver Nuggets
    "Little Caesars Arena": 20332,   # Detroit Pistons
    "Chase Center": 18064,           # Golden State Warriors
    "Toyota Center": 18055,          # Houston Rockets
    "Gainbridge Fieldhouse": 17923,  # Indiana Pacers
    "Crypto.com Arena": 19068,       # LA Clippers / Lakers
    "FedExForum": 17794,             # Memphis Grizzlies
    "Kaseya Center": 19600,          # Miami Heat
    "Fiserv Forum": 17341,           # Milwaukee Bucks
    "Target Center": 18024,          # Minnesota Timberwolves
    "Smoothie King Center": 16867,   # New Orleans Pelicans
    "Madison Square Garden": 19812,  # New York Knicks
    "Paycom Center": 18203,          # Oklahoma City Thunder
    "Kia Center": 18846,             # Orlando Magic
    "Wells Fargo Center": 20478,     # Philadelphia 76ers
    "Footprint Center": 17071,       # Phoenix Suns
    "Moda Center": 19441,            # Portland Trail Blazers
    "Golden 1 Center": 17608,        # Sacramento Kings
    "Frost Bank Center": 18418,      # San Antonio Spurs
    "Scotiabank Arena": 19800,       # Toronto Raptors
    "Delta Center": 18206,           # Utah Jazz
    "Capital One Arena": 20356,      # Washington Wizards
    
    # Older names/Previous Arenas
    "Staples Center": 19068,         # Old name for Crypto.com
    "Pepsi Center": 19605,           # Old name for Ball Arena
    "Quicken Loans Arena": 19432,    # Old name for Rocket Mortgage
    "Gund Arena": 19432,             # Older name for Rocket Mortgage
    "Philips Arena": 16600,          # Old name for State Farm
    "Time Warner Cable Arena": 19077,# Old name for Spectrum Center
    "Charlotte Bobcats Arena": 19077,# Old name for Spectrum Center
    "Amway Center": 18846,           # Old name for Kia Center
    "AT&T Center": 18418,            # Old name for Frost Bank Center
    "Talking Stick Resort Arena": 17071, # Old name for Footprint Center
    "Phoenix Suns Arena": 17071,     # Old name for Footprint Center
    "Bankers Life Fieldhouse": 17923,# Old name for Gainbridge
    "Chesapeake Energy Arena": 18203,# Old name for Paycom
    "Vivint Arena": 18206,           # Old name for Delta Center
    "Vivint Smart Home Arena": 18206,# Old name for Delta Center
    "EnergySolutions Arena": 18206,  # Old name for Delta Center
    "AmericanAirlines Arena": 19600, # Old name for Kaseya Center
    "Air Canada Centre": 19800,      # Old name for Scotiabank Arena
    "Verizon Center": 20356,         # Old name for Capital One
    "MCI Center": 20356,             # Older name for Capital One
    "Xfinity Mobile Arena": 20478,     # Old name for Wells Fargo Center
    "crypto.com Arena": 20000,       # Old name for Crypto.com Arena
    "Mortgage Matchup Center": 18422, # Temporary name for Rocket Mortgage FieldHouse during 2023-24 season
    "Toyota Center (Houston)": 18500, # To avoid confusion with Toyota Center in Kennewick, WA (used for some preseason games)
    "Intuit Dome": 18300,             # Future home of the LA Clippers (opening in 2024-25)
    
    # Historical Arenas requested
    "The Palace of Auburn Hills": 22076,
    "Oracle Arena": 19596,
    "Sleep Train Arena": 17317,
    "BMO Harris Bradley Center": 18717,
    "Bercy Arena": 15609,            # NBA Paris Games
    "Accor Arena": 15609,            # Modern name for Bercy
    "Benchmark International Arena": 20500, # 2020-21 Raptors temporary home (Amalie Arena)
}

In [4]:
def get_game_data(game_id):
    # Use internal API for cleaner data
    url = f"https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba/summary?event={game_id}"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            return None
            
        data = response.json()
        
        # Initialize
        game_result = {
            'game_id': game_id,
            'date': None,
            'opponent': None,
            'city': None,
            'state': None, 
            'arena': None,
            'attendance': None,
            'capacity': None,
            'attendance_percent': None,
            'hornets_wins': None,
            'hornets_losses': None,
            'opponent_wins': None,
            'opponent_losses': None,
            'hornets_score': None,
            'opponent_score': None,
            'location': None,
            # Leaders
            'hornets_top_scorer (points)': None,
            'opponents_top_scorer': None,
            'hornets_most_assists': None,
            'opponents_most_assists': None,
            'hornets_top_rebounder': None,
            'opponents_top_rebounder': None
        }
        
        # Game Info
        if 'gameInfo' in data:
            info = data['gameInfo']
            att = info.get('attendance')
            game_result['attendance'] = att
            
            if 'venue' in info:
                venue = info['venue']
                arena_name = venue.get('fullName')
                game_result['arena'] = arena_name
                if 'address' in venue:
                    game_result['city'] = venue['address'].get('city')
                    game_result['state'] = venue['address'].get('state')
                
                # Capacity Logic
                if arena_name in ARENA_CAPACITIES:
                    cap = ARENA_CAPACITIES[arena_name]
                    game_result['capacity'] = cap
                    if att and cap > 0:
                        pct = (att / cap) * 100
                        game_result['attendance_percent'] = round(pct, 1)
        
        # Teams and Records
        if 'header' in data and 'competitions' in data['header']:
            competition = data['header']['competitions'][0]
            
            # Check if game is strictly completed
            status = competition.get('status', {}).get('type', {})
            if not status.get('completed', False):
                return None
            
            # Format Date
            date_str = competition.get('date')
            if date_str:
                try:
                    dt_obj = datetime.strptime(date_str, "%Y-%m-%dT%H:%MZ")
                    game_result['date'] = dt_obj.strftime("%m/%d/%Y")
                except ValueError:
                    game_result['date'] = date_str[:10]
            
            for competitor in competition.get('competitors', []):
                team = competitor.get('team', {})
                name = team.get('displayName', '')
                
                # Format Record
                record_raw = competitor.get('record', [{}])[0].get('summary', 'N/A') if competitor.get('record') else 'N/A'
                
                # Extract wins/losses
                wins = None
                losses = None
                if record_raw != 'N/A' and '-' in record_raw:
                    parts = record_raw.split('-')
                    if len(parts) >= 2:
                        wins = parts[0]
                        losses = parts[1]
                
                score = competitor.get('score')
                
                if 'Hornets' in name:
                    game_result['hornets_wins'] = wins
                    game_result['hornets_losses'] = losses
                    game_result['hornets_score'] = score
                    home_away_status = competitor.get('homeAway', '')
                    game_result['location'] = 'Home' if home_away_status == 'home' else 'Away'
                else:
                    game_result['opponent'] = name
                    game_result['opponent_wins'] = wins
                    game_result['opponent_losses'] = losses
                    game_result['opponent_score'] = score

        # Leaders
        if 'leaders' in data:
            for team_leaders in data['leaders']:
                team_name = team_leaders.get('team', {}).get('displayName', '')
                is_hornets = 'Hornets' in team_name
                
                for category in team_leaders.get('leaders', []):
                    cat_name = category.get('name', '')
                    leader_list = category.get('leaders', [])
                    
                    if leader_list:
                        leader = leader_list[0]
                        athlete = leader.get('athlete', {})
                        player_name = athlete.get('displayName', '')
                        value = leader.get('displayValue', '')
                        
                        entry_val = f"{player_name} ({value})" 
                        
                        if is_hornets:
                            if cat_name == 'points':
                                game_result['hornets_top_scorer (points)'] = entry_val
                            elif cat_name == 'assists':
                                game_result['hornets_most_assists'] = entry_val
                            elif cat_name == 'rebounds':
                                game_result['hornets_top_rebounder'] = entry_val
                        else:
                            if cat_name == 'points':
                                game_result['opponents_top_scorer'] = entry_val
                            elif cat_name == 'assists':
                                game_result['opponents_most_assists'] = entry_val
                            elif cat_name == 'rebounds':
                                game_result['opponents_top_rebounder'] = entry_val

        return game_result

    except Exception as e:
        print(f"Error fetching {game_id}: {e}")
        return None

In [5]:
# Load All-Stars Data
all_stars_df = pd.read_csv('data/nba_all_stars.csv')

# Create a lookup dictionary for (Year, Team) -> Count
# Group by Year and Team, then count the number of players
all_star_counts = all_stars_df.groupby(['Year', 'Team']).size().to_dict()

# Helper function to get count (handling potential team name mismatches if needed later)
def get_all_star_count(year, team_name):
    # Try exact match
    count = all_star_counts.get((year, team_name), 0)
    return count

print("All-Star data loaded successfully.")

All-Star data loaded successfully.


In [ ]:
all_games_data = []
# years = [2026] # Just 2026 for verify filter
years = range(2015, 2027)

for year in years:
    print(f"Fetching schedule for {year}...")
    game_ids = get_schedule(year)
    print(f"Found {len(game_ids)} games for {year}")
    
    for game_id in tqdm(game_ids, desc=f"Games {year}"):
        data = get_game_data(game_id)
        if data:
            data['season'] = year
            
            # Calculate Opponent All-Stars
            opponent_name = data.get('opponent')
            if opponent_name:
                count = all_star_counts.get((year, opponent_name), 0)
                
                # Simple fallback for LA/Los Angeles
                if count == 0 and "Los Angeles" in opponent_name:
                     alt_name = opponent_name.replace("Los Angeles", "LA")
                     count = all_star_counts.get((year, alt_name), 0)
                elif count == 0 and "LA" in opponent_name:
                     alt_name = opponent_name.replace("LA", "Los Angeles")
                     count = all_star_counts.get((year, alt_name), 0)
                     
                data['opponent_all_stars_count'] = count
            else:
                data['opponent_all_stars_count'] = 0
                
            all_games_data.append(data)
        time.sleep(0.05)
            
# Preview the data (first 10 lines)
df_preview = pd.DataFrame(all_games_data)
# Check dates
print(df_preview['date'].max())
print(df_preview['date'].min())

Fetching schedule for 2026...
Found 82 games for 2026


Games 2026: 100%|██████████| 82/82 [00:52<00:00,  1.57it/s]

12/31/2025
01/03/2026


In [7]:
pd.set_option('display.max_columns', None)
df_preview.head()

,game_id,date,opponent,city,state,arena,attendance,capacity,attendance_percent,hornets_wins,hornets_losses,opponent_wins,opponent_losses,hornets_score,opponent_score,location,hornets_top_scorer (points),opponents_top_scorer,hornets_most_assists,opponents_most_assists,hornets_top_rebounder,opponents_top_rebounder,season,opponent_all_stars_count
0,401810567,02/02/2026,New Orleans Pelicans,Charlotte,NC,Spectrum Center,17263,19077.0,90.5,23,28,13,39,102,95,Home,LaMelo Ball (24),Trey Murphy III (27),LaMelo Ball (5),Saddiq Bey (5),Kon Knueppel (9),Zion Williamson (11),2026,0
1,401810806,03/12/2026,Sacramento Kings,Sacramento,CA,Golden 1 Center,15597,17608.0,88.6,34,33,16,51,117,109,Away,LaMelo Ball (30),DeMar DeRozan (39),LaMelo Ball (5),Nique Clifford (8),Moussa Diabate (10),Precious Achiuwa (8),2026,0
2,401810023,11/05/2025,New Orleans Pelicans,New Orleans,LA,Smoothie King Center,15555,16867.0,92.2,3,5,1,6,112,116,Away,Miles Bridges (22),Trey Murphy III (21),Collin Sexton (5),Derik Queen (7),Kon Knueppel (12),Derik Queen (8),2026,0
3,401809813,11/29/2025,Chicago Bulls,Charlotte,NC,Spectrum Center,19453,19077.0,102.0,5,14,9,9,123,116,Home,Brandon Miller (27),Josh Giddey (25),LaMelo Ball (8),Josh Giddey (9),Miles Bridges (8),Nikola Vucevic (14),2026,0
4,401810749,03/05/2026,Boston Celtics,Boston,MA,TD Garden,19156,19156.0,100.0,32,31,41,21,118,89,Away,Kon Knueppel (20),Derrick White (29),Coby White (6),Jaylen Brown (7),Moussa Diabate (9),Jaylen Brown (11),2026,1


In [9]:
# Final Save to local repository
output_path = 'data/hornets_games_2015-2026.csv'
df = pd.DataFrame(all_games_data)
df.to_csv(output_path, index=False)
print(f"Done! Saved to {output_path}")
print(df.head())
print(df.columns)

Done! Saved to data/hornets_games_2015-2026.csv
     game_id        date                opponent         city state  \
0  400578776  01/03/2015     Cleveland Cavaliers    Charlotte    NC   
1  400579337  03/22/2015  Minnesota Timberwolves  Minneapolis    MN   
2  400578701  12/23/2014          Denver Nuggets    Charlotte    NC   
3  400578349  11/06/2014              Miami Heat    Charlotte    NC   
4  400579352  03/25/2015           Brooklyn Nets    Charlotte    NC   

             arena  attendance  capacity  attendance_percent hornets_wins  \
0  Spectrum Center       19307   19077.0               101.2           10   
1    Target Center       15262   18024.0                84.7           30   
2  Spectrum Center       16913   19077.0                88.7            9   
3  Spectrum Center       15874   19077.0                83.2            2   
4  Spectrum Center       15091   19077.0                79.1           30   

  hornets_losses opponent_wins opponent_losses hornets_score o